In [2]:
import glob, os, sys, math, json
import numpy as np
import pandas as pd
import torch
import transformers

DOSSIER = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
sys.path.append(DOSSIER)
from metrics import GROUPES, evaluer, rapport_par_groupe, probas_vers_predictions, chercher_seuils

train = pd.read_csv(f"{DOSSIER}/train.csv", dtype={"id": str})
val = pd.read_csv(f"{DOSSIER}/val.csv", dtype={"id": str})

print("Train :", train.shape, "| Validation :", val.shape)
print("transformers :", transformers.__version__)
print("GPU :", torch.cuda.device_count(), "x",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN")

Train : (12132, 11) | Validation : (2609, 11)
transformers : 5.20.2
GPU : 2 x Tesla T4


In [3]:
from datasets import Dataset, Sequence, Value
from transformers import AutoTokenizer

NOM_MODELE = "allenai/scibert_scivocab_uncased"
MAX_LENGTH = 384   # choisi à l'étape 2 : 97,9 % des résumés lus en entier

tokenizer = AutoTokenizer.from_pretrained(NOM_MODELE)

def vers_dataset(df):
    """Tableau pandas -> Dataset Hugging Face : textes tokenisés + 8 labels en float32."""
    ds = Dataset.from_dict({
        "text": df["text"].tolist(),
        "labels": df[GROUPES].values.astype("float32").tolist(),
    })
    ds = ds.cast_column("labels", Sequence(Value("float32")))
    return ds.map(lambda lot: tokenizer(lot["text"], truncation=True, max_length=MAX_LENGTH),
                  batched=True, remove_columns=["text"])

ds_train = vers_dataset(train)
ds_val = vers_dataset(val)
print(ds_train)
print("Exemple de tokens :", tokenizer.convert_ids_to_tokens(ds_train[0]["input_ids"][:15]))

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/228k [00:00<?, ?B/s]

Casting the dataset:   0%|          | 0/12132 [00:00<?, ? examples/s]

Map:   0%|          | 0/12132 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/2609 [00:00<?, ? examples/s]

Map:   0%|          | 0/2609 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 12132
})
Exemple de tokens : ['[CLS]', 'in', 'this', 'note', 'we', 'develop', 'tools', 'and', 'techniques', 'for', 'the', 'treatment', 'of', 'anisotropic', 'thermo']


In [4]:
from transformers import AutoModelForSequenceClassification

modele = AutoModelForSequenceClassification.from_pretrained(
    NOM_MODELE,
    num_labels=len(GROUPES),                     # 8 sorties
    problem_type="multi_label_classification",   # sigmoïde + perte BCE, comme tes BiLSTM
    id2label=dict(enumerate(GROUPES)),
    label2id={g: i for i, g in enumerate(GROUPES)},
)
print("Nombre de paramètres :", sum(p.numel() for p in modele.parameters()))

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  442MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

[transformers] BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those

Nombre de paramètres : 109924616


In [5]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

def calculer_metriques(eval_pred):
    """Appelée après chaque epoch : scores sur la validation (seuil 0.5), avec TON metrics.py."""
    logits, labels = eval_pred
    probas = 1 / (1 + np.exp(-logits))   # sigmoïde : scores -> probabilités
    return evaluer(labels.astype(int), probas_vers_predictions(probas))

NB_EPOCHS = 4
TAILLE_LOT = 16                           # par GPU (donc 32 au total avec 2 GPU)
nb_gpus = max(1, torch.cuda.device_count())
pas_par_epoch = math.ceil(len(ds_train) / (TAILLE_LOT * nb_gpus))
pas_warmup = int(0.1 * pas_par_epoch * NB_EPOCHS)   # 10 % des pas d'entraînement
print("Pas par epoch :", pas_par_epoch, "| pas de warmup :", pas_warmup)

args = TrainingArguments(
    output_dir="/kaggle/working/scibert",
    num_train_epochs=NB_EPOCHS,
    learning_rate=2e-5,                   # très petit : on ajuste SciBERT sans casser ce qu'il sait
    per_device_train_batch_size=TAILLE_LOT,
    per_device_eval_batch_size=64,
    warmup_steps=pas_warmup,              # le learning rate monte doucement au début
    weight_decay=0.01,
    fp16=True,                            # calculs en demi-précision : environ 2x plus rapide sur T4
    eval_strategy="epoch",                # note sur la validation à la fin de chaque epoch
    save_strategy="epoch",
    load_best_model_at_end=True,          # recharge la meilleure epoch (comme ton early stopping)
    metric_for_best_model="micro_f1",
    save_total_limit=1,
    logging_strategy="epoch",
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=modele,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=DataCollatorWithPadding(tokenizer),  # padding dynamique, paquet par paquet
    compute_metrics=calculer_metriques,
)

trainer.train()

Pas par epoch : 380 | pas de warmup : 152


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Hamming Loss,Subset Accuracy,Runtime,Samples Per Second,Steps Per Second
1,0.638110,0.427314,0.790488,0.786394,0.082311,0.543503,30.704000,84.973000,0.684000
2,0.364957,0.376076,0.814131,0.811690,0.073352,0.585282,30.622800,85.198000,0.686000
3,0.296536,0.370397,0.820501,0.818079,0.072825,0.592181,30.499900,85.541000,0.689000
4,0.251952,0.376168,0.817484,0.814985,0.074023,0.584898,30.686000,85.022000,0.684000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1520, training_loss=0.3878886574193051, metrics={'train_runtime': 1871.9216, 'train_samples_per_second': 25.924, 'train_steps_per_second': 0.812, 'total_flos': 8801025012585600.0, 'train_loss': 0.3878886574193051, 'epoch': 4.0})

In [6]:
# Évaluation sur la VALIDATION avec la meilleure epoch (rechargée automatiquement par le Trainer)
pred_val = trainer.predict(ds_val)
probas_val = 1 / (1 + np.exp(-pred_val.predictions))
y_val = pred_val.label_ids.astype(int)

scores_05 = evaluer(y_val, probas_vers_predictions(probas_val))
seuils = chercher_seuils(y_val, probas_val)
scores_seuils = evaluer(y_val, probas_vers_predictions(probas_val, seuils))

print("Seuils choisis :", dict(zip(GROUPES, seuils.tolist())))
print(f"{'':16s} {'seuil 0.5':>10s} {'seuils/groupe':>14s}")
for nom in scores_05:
    print(f"{nom:16s} {scores_05[nom]:>10.4f} {scores_seuils[nom]:>14.4f}")
print()
print(rapport_par_groupe(y_val, probas_vers_predictions(probas_val, seuils)))

with open("/kaggle/working/scibert_val.json", "w") as f:
    json.dump({"seuil_05": scores_05, "seuils": seuils.tolist(), "seuils_par_groupe": scores_seuils,
               "historique": trainer.state.log_history}, f, indent=2)
np.save("/kaggle/working/scibert_val_probas.npy", probas_val)
print("Sauvegardé : scibert_val.json, scibert_val_probas.npy")

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Seuils choisis : {'cs': 0.55, 'math': 0.25, 'physics': 0.3, 'q-bio': 0.55, 'q-fin': 0.3, 'stat': 0.45, 'eess': 0.45, 'econ': 0.35}
                  seuil 0.5  seuils/groupe
micro_f1             0.8205         0.8278
macro_f1             0.8181         0.8272
hamming_loss         0.0728         0.0726
subset_accuracy      0.5922         0.5845

              precision    recall  f1-score   support

          cs       0.86      0.85      0.85      1023
        math       0.73      0.82      0.78       606
     physics       0.82      0.87      0.85       623
       q-bio       0.88      0.90      0.89       389
       q-fin       0.84      0.91      0.87       436
        stat       0.75      0.73      0.74       484
        eess       0.82      0.81      0.81       383
        econ       0.77      0.88      0.82       366

   micro avg       0.81      0.85      0.83      4310
   macro avg       0.81      0.85      0.83      4310
weighted avg       0.81      0.85      0.83      4310
 sa

In [1]:
import shutil

# Prédictions sur le TEST : on les SAUVEGARDE sans calculer de score.
# Le score test sera calculé UNE SEULE FOIS, à l'étape 8, pour tous les modèles ensemble.
test = pd.read_csv(f"{DOSSIER}/test.csv", dtype={"id": str})
ds_test = Dataset.from_dict({"text": test["text"].tolist()}).map(
    lambda lot: tokenizer(lot["text"], truncation=True, max_length=MAX_LENGTH),
    batched=True, remove_columns=["text"])            # pas de labels -> aucun score calculé
pred_test = trainer.predict(ds_test)
probas_test = 1 / (1 + np.exp(-pred_test.predictions))
np.save("/kaggle/working/scibert_test_probas.npy", probas_test)
test[["id"]].to_csv("/kaggle/working/scibert_test_ids.csv", index=False)
print("Probabilités test sauvegardées :", probas_test.shape)

# Sauvegarde du modèle final + nettoyage des checkpoints intermédiaires
trainer.save_model("/kaggle/working/scibert_final")
tokenizer.save_pretrained("/kaggle/working/scibert_final")
shutil.rmtree("/kaggle/working/scibert", ignore_errors=True)
print("Modèle sauvegardé dans /kaggle/working/scibert_final")

NameError: name 'pd' is not defined